# Model Training Pipeline

In [7]:
!pip3 install sagemaker

  Using cached sagemaker-3.4.1-py3-none-any.whl.metadata (20 kB)
Using cached sagemaker-3.4.1-py3-none-any.whl (11 kB)


In [8]:
import pandas as pd
import numpy as np
import sagemaker
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import pandas as pd
import boto3
from time import gmtime, strftime
import re
import time

In [3]:
import boto3

s3 = boto3.client("s3")

response = s3.list_objects_v2(
    Bucket="sagemaker-us-east-1-318401170150",
    Prefix="feature-store/buoy/318401170150/sagemaker/us-east-1/offline-store/buoy-wave-features-v2-1770682984/data"
)

In [4]:
from pyathena import connect

conn = connect(
    s3_staging_dir="s3://sagemaker-us-east-1-318401170150/athena-query-results/",
    region_name="us-east-1",
)

In [5]:
query = """
SELECT *
FROM AwsDataCatalog.sagemaker_featurestore.buoy_wave_features_v2_1770682984
WHERE target_E_star IS NOT NULL
LIMIT 5
"""

df = pd.read_sql(query, conn)
df.head()

/tmp/ipykernel_1505/3077023716.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,station_id,wind_direction,wind_speed,wind_gust,wave_height,dominant_wave_period,average_wave_period,mean_wave_direction,pressure,air_temperature,...,dominant_wave_period_lag_3,wave_height_lag_6,e_star_lag_6,wind_speed_lag_6,dominant_wave_period_lag_6,event_time,record_id,write_time,api_invocation_time,is_deleted
0,46086,None,None,None,None,None,None,None,None,None,...,12.12,3.74,13.9876,8.8,12.12,2023-01-17T03:40:00Z,46086_2023-01-17T03:40:00Z,2026-02-10 01:01:58.992,2026-02-10 00:57:23,False
1,46086,None,None,None,None,None,None,None,None,None,...,10.00,3.58,12.8164,10.4,11.43,2023-01-02T12:40:00Z,46086_2023-01-02T12:40:00Z,2026-02-10 01:02:01.106,2026-02-10 00:56:59,False
2,46086,None,None,None,None,None,None,None,None,None,...,10.00,3.58,12.8164,10.4,11.43,2023-01-02T12:40:00Z,46086_2023-01-02T12:40:00Z,2026-02-10 01:02:01.106,2026-02-10 00:57:17,False
3,46086,None,None,None,None,None,None,None,None,None,...,16.00,4.76,22.6576,4.1,17.39,2023-01-06T20:10:00Z,46086_2023-01-06T20:10:00Z,2026-02-10 01:02:00.438,2026-02-10 00:57:01,False
4,46086,None,None,None,None,None,None,None,None,None,...,16.00,4.76,22.6576,4.1,17.39,2023-01-06T20:10:00Z,46086_2023-01-06T20:10:00Z,2026-02-10 01:02:00.438,2026-02-10 00:57:18,False


In [6]:
df.columns

Index(['station_id', 'wind_direction', 'wind_speed', 'wind_gust',
       'wave_height', 'dominant_wave_period', 'average_wave_period',
       'mean_wave_direction', 'pressure', 'air_temperature',
       'water_temperature', 'dewpoint_temperature', 'wind_speed_ms',
       'wave_energy', 'e_star', 'target_e_star', 'wave_height_lag_1',
       'e_star_lag_1', 'wind_speed_lag_1', 'dominant_wave_period_lag_1',
       'wave_height_lag_2', 'e_star_lag_2', 'wind_speed_lag_2',
       'dominant_wave_period_lag_2', 'wave_height_lag_3', 'e_star_lag_3',
       'wind_speed_lag_3', 'dominant_wave_period_lag_3', 'wave_height_lag_6',
       'e_star_lag_6', 'wind_speed_lag_6', 'dominant_wave_period_lag_6',
       'event_time', 'record_id', 'write_time', 'api_invocation_time',
       'is_deleted'],
      dtype='object')

In [7]:
feature_cols = [c for c in df.columns if c not in ["target_E_star", "record_id", "event_time",
                                                   "write_time", "api_invocation_time", "is_deleted"]]
target_col = "target_e_star"

In [8]:
n = len(df)
train_end = int(n * 0.8)
val_end = int(n * 0.9)

X = df[feature_cols]
y = df[target_col]

X_train, y_train = X.iloc[:train_end], y.iloc[:train_end]
X_val, y_val = X.iloc[train_end:val_end], y.iloc[train_end:val_end]
X_test, y_test = X.iloc[val_end:], y.iloc[val_end:]

print(f"Train: {len(X_train)} rows, Val: {len(X_val)} rows, Test: {len(X_test)} rows")

Train: 4 rows, Val: 0 rows, Test: 1 rows


In [15]:
train_path = f"s3://{bucket}/{prefix}/train/train.csv"
val_path = f"s3://{bucket}/{prefix}/validation/validation.csv"
test_path = f"s3://{bucket}/{prefix}/test/test.csv"

X_train.assign(target_E_star=y_train).to_csv(train_path, index=False, header=True)
X_val.assign(target_E_star=y_val).to_csv(val_path, index=False, header=True)
X_test.assign(target_E_star=y_test).to_csv(test_path, index=False, header=True)

NameError: name 'bucket' is not defined

In [10]:
from sklearn.ensemble import HistGradientBoostingRegressor
model = HistGradientBoostingRegressor(
    max_iter=400,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)

print("Training model...")
model.fit(X_train, y_train)


Training model...


,loss,'squared_error'
,quantile,None
,learning_rate,0.05
,max_iter,400
,max_leaf_nodes,31
,max_depth,6
,min_samples_leaf,20
,l2_regularization,0.0
,max_features,1.0
,max_bins,255
,categorical_features,'from_dtype'


In [13]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
print(f"Test RMSE: {rmse:.3f}, MAE: {mae:.3f}")

Test RMSE: 4.612, MAE: 4.612


In [14]:
import joblib
import os

model_dir = "../models"
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, "buoy_wave_model.joblib")
joblib.dump(model, model_path)
print(f"Model saved to {model_path}")

Model saved to models/buoy_wave_model.joblib


In [32]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>